In [ ]:
mo.md(
    f"""
    # Unsloth v5 10-Way Comparison Notebook

    Compare **10 backends** on any of the 6 BIEP LC subjects' PDFs:

    - **6 Unsloth VLMs** (served via unsloth-serve :8889 — `local/unsloth/*` LiteLLM routes)
    - **4 Classical OCRs** (Docling, dots-ocr, OlmOCR, PaddleOCR)

    All 10 backends route through the canonical OCR-Router at
    `http://ocr-router:8090/v1/ocr` (the same endpoint the 6 BIEP v3
    jurisdiction dashboards use).

    **Human-driven only** — no Dagster asset wiring. Exports to
    `stedding/eval_results/unsloth_compare_{model_role}_{pdf_hash}.json`
    for future BIEP v2 ingestion.

    Registry: **{_REGISTRY_SUMMARY.get("total", 0)}** total models / **{_REGISTRY_SUMMARY.get("available", 0)}** available.
    Default LLM: `{_DEFAULT_LLM}`.
    """
)

In [ ]:
"""The 10 backend picker (6 Unsloth VLMs + 4 Classical OCRs)."""
# The 6 Unsloth VLMs (from MODEL_REGISTRY, filtered by backend="unsloth"
# + family="ocr_vision"). Each entry's litellm_alias is the canonical
# route added in phase 3 (local/unsloth/<key>).
unsloth_vlm_keys = [
    e.key
    for e in MODEL_REGISTRY.filter(family="ocr_vision")
    if (e.backend == "unsloth" and "Qwen3-VL" in e.upstream_id)
    or "GLM-4" in e.upstream_id
    or "DeepSeek-OCR" in e.upstream_id
]
# Sorted for deterministic ordering
unsloth_vlms = sorted(set(unsloth_vlm_keys))

# The 4 classical OCR backends (the BIEP v2 4-path ensemble).
# These are the entries in MODEL_REGISTRY that route through the
# ocr-router :8090 dispatch matrix (not through litellm).
classical_ocr = [
    "docling-serve",  # IBM Docling HTTP REST API
    "dots-ocr",  # layout specialist
    "olmocr",  # tables + latex
    "paddleocr",  # multilingual
]

all_10_backends = unsloth_vlms + classical_ocr

backend_picker = mo.ui.multiselect(
    options=all_10_backends,
    value=unsloth_vlms[:3] + classical_ocr[:1],  # default: 3 unsloth + 1 classical
    label="10 Backends (6 Unsloth VLMs + 4 Classical OCRs)",
)

capability_filter = mo.ui.multiselect(
    options=[
        "DENSE_OCR",
        "GROUNDING",
        "TABLES",
        "LATEX",
        "REASONING",
        "MULTILINGUAL",
        "GAELIC",
        "DIAGRAM",
    ],
    value=[],
    label="Filter by Model Capability (optional)",
)

mo.vstack([backend_picker, capability_filter])

In [ ]:
"""The PDF picker (6 BIEP LC subjects from stedding/ingest_queue/)."""
# The 6 BIEP LC subjects (per the british-isles-education-pipeline-v3 spec).
# The PDF picker reads from stedding/ingest_queue/ (the canonical
# location for the 6 subjects' PDFs). The list_pipelines helper
# enumerates the available PDFs.
try:
    from notebooks._shared.schema import list_pipelines

    pdf_paths = list_pipelines()
except ImportError:
    pdf_paths = []

# If the helper is not available, fall back to the canonical 6 BIEP
# subjects. The PDFs live at stedding/ingest_queue/{subject}/*.pdf
pdf_picker = mo.ui.multiselect(
    options=pdf_paths
    or [
        "stedding/ingest_queue/mathematics/lc_2024_paper_1.pdf",
        "stedding/ingest_queue/chemistry/lc_2024_paper_1.pdf",
        "stedding/ingest_queue/geography/lc_2024_paper_1.pdf",
        "stedding/ingest_queue/gaeilge/lc_2024_paper_1.pdf",
        "stedding/ingest_queue/english/lc_2024_paper_1.pdf",
        "stedding/ingest_queue/computer_science/lc_2024_paper_1.pdf",
    ],
    value=[],
    label="PDFs (from stedding/ingest_queue/)",
)

In [ ]:
"""The export button (writes results to stedding/eval_results/)."""
export_button = mo.ui.run_button(label="Export results to stedding/eval_results/")

In [ ]:
"""The side-by-side comparison reactive loop.

For each (backend, PDF) pair, sends a request to the OCR-Router at
http://ocr-router:8090/v1/ocr. The router fans out to the canonical
backends (classical OCRs + llama-swap + unsloth-serve).

Each column shows:
- Model name + backend (unsloth / classical)
- Response text (for VLMs) / DocTags XML (for Docling) / regions (for OCRs)
- Latency (ms)
- Tokens (VLMs) / regions (classical)
- KL-divergence note (for VLMs) / CER/WER note (for classical)
"""
import hashlib
import json
import time
from pathlib import Path

backends = backend_picker.value if backend_picker else []
pdfs = pdf_picker.value if pdf_picker else []

if not backends or not pdfs:
    mo.md("> Select at least 1 backend and 1 PDF to see results.")
    return

# The canonical OCR-Router endpoint (the same one the 6 BIEP v3
# jurisdiction dashboards use). The router handles the fanout to
# the 4 classical OCRs + the 6 unsloth-served VLMs.
ocr_router_url = "http://ocr-router:8090/v1/ocr"

results = []
for pdf_path in pdfs:
    for backend in backends:
        # The OCR-Router dispatch: model_name = backend key,
        # file_path = PDF path. The router handles the backend-specific
        # input/output format (e.g. DocTags XML for Docling, plain text
        # for VLMs, regions for OCRs).
        started = time.time()
        try:
            # The canonical POST envelope (per the ocr-router contract).
            # The router does the model_id → backend fanout internally.
            import urllib.error
            import urllib.request

            payload = json.dumps(
                {
                    "model": backend,
                    "file_path": pdf_path,
                    "stream": False,
                }
            ).encode("utf-8")
            req = urllib.request.Request(
                ocr_router_url,
                data=payload,
                headers={"Content-Type": "application/json"},
                method="POST",
            )
            with urllib.request.urlopen(req, timeout=600) as resp:
                response = json.loads(resp.read().decode("utf-8"))
            latency_ms = int((time.time() - started) * 1000)
            results.append(
                {
                    "backend": backend,
                    "pdf": pdf_path,
                    "latency_ms": latency_ms,
                    "response": response,
                    "status": "ok",
                }
            )
        except (
            urllib.error.URLError,
            urllib.error.HTTPError,
            TimeoutError,
            json.JSONDecodeError,
        ) as exc:
            latency_ms = int((time.time() - started) * 1000)
            results.append(
                {
                    "backend": backend,
                    "pdf": pdf_path,
                    "latency_ms": latency_ms,
                    "response": str(exc),
                    "status": "error",
                }
            )

# Side-by-side rendering: hstack of vstack columns, one per backend.
columns = []
for backend in backends:
    backend_results = [r for r in results if r["backend"] == backend]
    col = mo.vstack(
        [
            mo.md(f"### `{backend}`"),
            mo.md(
                "**Backend:** "
                + (
                    "Unsloth Studio"
                    if backend
                    in [
                        e.key
                        for e in MODEL_REGISTRY.filter(family="ocr_vision")
                        if e.backend == "unsloth"
                    ]
                    else "Classical OCR"
                )
            ),
            *[
                mo.md(
                    f"**PDF:** `{Path(r['pdf']).name}`\n"
                    f"**Latency:** {r['latency_ms']} ms\n"
                    f"**Status:** {r['status']}\n\n"
                    f"```\n{str(r['response'])[:1500]}\n```"
                )
                for r in backend_results
            ],
        ]
    )
    columns.append(col)

side_by_side = mo.hstack(columns, gap=2)

# Export logic (per the human-driven contract — no Dagster asset).
if export_button.value:
    export_dir = Path("stedding/eval_results")
    export_dir.mkdir(parents=True, exist_ok=True)
    for r in results:
        pdf_hash = hashlib.sha256(r["pdf"].encode("utf-8")).hexdigest()[:12]
        pdf_name = Path(r["pdf"]).stem
        export_path = export_dir / f"unsloth_compare_{r['backend']}_{pdf_name}_{pdf_hash}.json"
        export_path.write_text(json.dumps(r, indent=2, default=str))
    export_md = mo.md(f"> ✅ Exported **{len(results)}** results to `stedding/eval_results/`.")
else:
    export_md = mo.md("> Click **Export** to save results to `stedding/eval_results/`.")

mo.vstack([side_by_side, export_md])